# ROGII v2.8: Beam Search + Particle Filters + v2.1 Ensemble

Blend: 0.5*v2.1 + 0.25*Beam + 0.25*PF

In [ ]:
import subprocess, sys
for p in ["numba"]:
    if subprocess.run([sys.executable, "-m", "pip", "show", p], capture_output=True).returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", p, "--quiet"])

from pathlib import Path
from scipy.interpolate import interp1d
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from numba import njit
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


def _find_data():
    for p in [
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
    ]:
        if (p / "train").exists():
            return p
    raise FileNotFoundError("Competition data folder not found")


DATA_DIR = _find_data()
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
SAMPLE_SUB = pd.read_csv(DATA_DIR / "sample_submission.csv")
ID_SET = set(SAMPLE_SUB["id"].tolist())
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

print(f"Data: {DATA_DIR}")


## Beam Search

In [ ]:
@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, bs, move_cost, emit_scale):
    n, nt = len(sgr), len(tw_gr)
    max_c = bs * 5
    bidx = np.zeros(bs, np.int64)
    bidx[0] = si
    bcost = np.full(bs, 1e30)
    bcost[0] = 0.0
    bn = np.int64(1)

    hist_i = np.zeros((n, bs), np.int64)
    hist_p = np.zeros((n, bs), np.int64)

    c_i = np.zeros(max_c, np.int64)
    c_c = np.full(max_c, 1e30)
    c_p = np.zeros(max_c, np.int64)

    for t in range(n):
        gv = sgr[t]
        nc = np.int64(0)
        for bi in range(bn):
            idx, cost = bidx[bi], bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if 0 <= ni < nt:
                    tot = cost + (gv - tw_gr[ni]) ** 2 / emit_scale + move_cost * abs(d)
                    fnd = -1
                    for ci in range(nc):
                        if c_i[ci] == ni:
                            fnd = ci
                            break
                    if fnd >= 0:
                        if tot < c_c[fnd]:
                            c_c[fnd] = tot
                            c_p[fnd] = bi
                    elif nc < max_c:
                        c_i[nc], c_c[nc], c_p[nc] = ni, tot, bi
                        nc += 1

        kept = min(bs, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if c_c[j] < c_c[mi]:
                    mi = j
            if mi != i:
                c_i[i], c_i[mi] = c_i[mi], c_i[i]
                c_c[i], c_c[mi] = c_c[mi], c_c[i]
                c_p[i], c_p[mi] = c_p[mi], c_p[i]

        hist_i[t, :kept] = c_i[:kept]
        hist_p[t, :kept] = c_p[:kept]
        bidx[:kept] = c_i[:kept]
        bcost[:kept] = c_c[:kept]
        bn = kept

    best = 0
    for b in range(1, bn):
        if bcost[b] < bcost[best]:
            best = b

    path = np.zeros(n, np.int64)
    b = best
    for t in range(n - 1, -1, -1):
        path[t] = hist_i[t, b]
        b = hist_p[t, b]
    return path


def _nn(a, v):
    i = int(np.searchsorted(a, v, "left"))
    if i >= len(a):
        return len(a) - 1
    if i > 0 and abs(a[i - 1] - v) <= abs(a[i] - v):
        return i - 1
    return i


def beam_search(gr_hidden, tw_tvt, tw_gr, start_tvt):
    if len(gr_hidden) == 0:
        return np.array([], dtype=np.float32)
    si = _nn(tw_tvt, start_tvt)
    path = _beam_jit(gr_hidden.astype(np.float64), tw_gr.astype(np.float64), si, 10, 20.0, 144.0)
    return tw_tvt[path].astype(np.float32)


_ = _beam_jit(np.random.randn(30), np.random.randn(50), 20, 8, 10.0, 100.0)
print("Beam ready")


## Particle Filter

In [ ]:
def run_pf_z(full_hw, tw_tvt, tw_gr, n=250):
    known = full_hw[full_hw["TVT_input"].notna()]
    hidden = full_hw[full_hw["TVT_input"].isna()]
    if len(hidden) == 0 or len(known) == 0:
        return np.array([], dtype=np.float32)

    tmin, tmax = tw_tvt.min(), tw_tvt.max()
    tf_gr = interp1d(tw_tvt, tw_gr, bounds_error=False, fill_value=(tw_gr[0], tw_gr[-1]))

    pos = float(known["TVT_input"].iloc[-1]) + np.random.normal(0, 0.5, n)
    vel = np.random.normal(0, 0.02, n)
    w = np.ones(n) / n

    out = []
    md0 = float(known["MD"].iloc[-1])
    for _, row in hidden.iterrows():
        dm = max(float(row["MD"]) - md0, 1.0)
        vel = 0.993 * vel + np.random.normal(0, 0.005, n)
        pos = np.clip(pos + vel * dm + np.random.normal(0, 0.01, n), tmin - 50, tmax + 50)

        gr_obs = row["GR"]
        if pd.notna(gr_obs):
            lk = np.exp(-0.5 * ((gr_obs - tf_gr(pos)) / 30.0) ** 2)
            lk = np.maximum(lk, 1e-300)
            w = w * lk
            sw = w.sum()
            w = w / sw if sw > 0 else np.ones(n) / n

        out.append(float(np.average(pos, weights=w)))
    return np.array(out, dtype=np.float32)


print("PF ready")


## Train + Inference

In [ ]:
def load_csv(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None


train_hw = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
test_hw = sorted(TEST_DIR.glob("*__horizontal_well.csv"))

train_data = {p.stem.replace("__horizontal_well", ""): load_csv(p) for p in train_hw}
train_data = {k: v for k, v in train_data.items() if v is not None}

tw_data = {}
for p in TRAIN_DIR.glob("*__typewell.csv"):
    w = p.stem.replace("__typewell", "")
    df = load_csv(p)
    if df is not None:
        tw_data[w] = df.sort_values("TVT")

# Compact baseline features
x_list, y_list = [], []
for _, hw in train_data.items():
    known = hw[hw["TVT_input"].notna()]
    if len(known) >= 5:
        x_list.append([
            known["MD"].mean(),
            known["Z"].mean(),
            known["GR"].fillna(known["GR"].median()).mean(),
            known["TVT_input"].mean(),
        ])
        y_list.append(known["TVT"].mean())

x_train = np.array(x_list, dtype=np.float32)
y_train = np.array(y_list, dtype=np.float32)

hgb = HGB(loss="squared_error", max_iter=300, random_state=SEED, n_iter_no_change=30)
hgb.fit(x_train, y_train)

ridge = Ridge(alpha=1.0, positive=True)
ridge.fit(np.nan_to_num(x_train), y_train)

print(f"Baseline ready: {len(x_train)} samples")

pred_ids = []
preds = []
well_seen = 0

for p in test_hw:
    wid = p.stem.replace("__horizontal_well", "")
    hw = load_csv(p)
    if hw is None or wid not in tw_data:
        continue

    known = hw[hw["TVT_input"].notna()]
    hidden = hw[hw["TVT_input"].isna()]
    if len(known) == 0 or len(hidden) == 0:
        continue

    well_seen += 1
    tw = tw_data[wid]
    tw_tvt = tw["TVT"].values.astype(np.float32)
    tw_gr = tw["GR"].fillna(tw["GR"].median()).values.astype(np.float32)

    xh = np.column_stack([
        hidden["MD"].values,
        hidden["Z"].values,
        hidden["GR"].fillna(hidden["GR"].median()).values,
        np.full(len(hidden), float(known["TVT_input"].iloc[-1])),
    ]).astype(np.float32)

    hgb_pred = hgb.predict(xh)
    ridge_pred = ridge.predict(np.nan_to_num(xh))
    v21 = 0.75 * hgb_pred + 0.25 * ridge_pred

    beam = beam_search(
        hidden["GR"].fillna(hidden["GR"].median()).values.astype(np.float32),
        tw_tvt,
        tw_gr,
        float(known["TVT_input"].iloc[-1]),
    )
    if len(beam) != len(hidden):
        beam = v21

    pf = run_pf_z(hw, tw_tvt, tw_gr, n=250)
    if len(pf) != len(hidden):
        pf = v21

    final = np.clip(0.5 * v21 + 0.25 * beam + 0.25 * pf, 11700.0, 12500.0)

    for idx, pv in zip(hidden.index, final):
        pid = f"{wid}_{idx}"
        if pid in ID_SET:
            pred_ids.append(pid)
            preds.append(float(pv))

sub = SAMPLE_SUB.copy()
sub["tvt"] = sub["id"].map(dict(zip(pred_ids, preds)))
sub["tvt"] = sub["tvt"].fillna(12000.0)

print(f"Processed wells: {well_seen}")
print(f"Generated predictions: {len(preds)}")
print(f"Output shape: {sub.shape}, missing: {sub['tvt'].isna().sum()}, finite: {np.isfinite(sub['tvt']).all()}")
print(f"Range: [{sub['tvt'].min():.3f}, {sub['tvt'].max():.3f}]")

sub.to_csv(OUT_DIR / "submission.csv", index=False)
print("Saved submission.csv")
